# NVIDIA RAG Blueprint - Quick Start Guide

This notebook walks you through deploying and using the NVIDIA RAG (Retrieval-Augmented Generation) system.

## What is RAG?

RAG is a technique that improves AI responses by:
1. **Storing** your documents as searchable vectors (like an index)
2. **Searching** for relevant content when you ask a question
3. **Generating** an answer using only the retrieved context

Think of it like `grep` for AI - it finds relevant text first, then answers based on what it found.

---

## Table of Contents

| Section | Description | Time |
|---------|-------------|------|
| **0. Setup** | Install dependencies, define helper functions | 2 min |
| **1. Prerequisites** | Check hardware/software requirements | 2 min |
| **2. Deploy** | Start all Docker containers | 10-15 min |
| **3. Test APIs** | Verify services are working | 5 min |
| **4. Use the Chatbot** | Upload docs, ask questions | 5 min |
| **5. Cleanup** | Stop containers, free resources | 1 min |

---

## ⚠️ Before You Start

- **GPU Required**: NVIDIA GPU(s) with 80GB+ VRAM total
- **API Key Required**: Get one from https://org.ngc.nvidia.com/setup/api-keys
- **Disk Space**: 200GB+ free for Docker images, NIM model cache, vector database data, logs, and temporary files
- **Memory**: Docker daemon memory must cover the selected local NIM services; the requirements check calculates this from the launchable service set

---

# Section 0: Setup

---

Run these cells once to set up the environment.

### 0.1 Install Required Python Packages

In [ ]:
# Install dependencies (run once)
import sys
try:
    import pip  # noqa
except ImportError:
    !{sys.executable} -m ensurepip --upgrade

!{sys.executable} -m pip install -q python-dotenv aiohttp requests

### 0.2 Load Helper Functions

All utility functions are defined in this cell. **Run it once before proceeding.**

In [ ]:
"""
NVIDIA RAG Blueprint - Helper Functions
=======================================
Run this cell once to load all utilities.
"""

import os
import sys
import json
import subprocess
import asyncio
import aiohttp
import base64
import re
from typing import List, Optional
from pathlib import Path
from IPython.display import display, Image as IPImage
from dotenv import load_dotenv


# =============================================================================
# CONFIGURATION - Edit these if services run on different hosts
# =============================================================================

IPADDRESS = "0.0.0.0"
RAG_SERVER_PORT = "8081"
INGESTOR_SERVER_PORT = "8082"
ELASTICSEARCH_ENDPOINT = "http://elasticsearch:9200"

RAG_BASE_URL = f"http://{IPADDRESS}:{RAG_SERVER_PORT}"
INGESTOR_BASE_URL = f"http://{IPADDRESS}:{INGESTOR_SERVER_PORT}"
ENV_FILE = Path("deploy/compose/.env")
LAUNCHABLE_ENV_BEGIN = "# >>> launchable.ipynb generated env >>>"
LAUNCHABLE_ENV_END = "# <<< launchable.ipynb generated env <<<"
LAUNCHABLE_ENV_VALUES = {}

# NIM services to deploy for the launchable default path.
# This intentionally excludes nim-llm and vlm-ms because the notebook configures
# hosted NVIDIA endpoints for generation. Starting all default nims.yaml services
# would try to launch the large local LLM and can exhaust Docker/GPU memory.
NIM_SERVICES = (
    "nemotron-embedding-ms",
    "nemotron-ranking-ms",
    "page-elements",
    "graphic-elements",
    "table-structure",
    "nemotron-ocr",
)
NIM_SERVICE_ARGS = " ".join(NIM_SERVICES)
# `nemotron-embedding-ms` lives under the `text-embed` profile in nims.yaml,
# so the docker compose calls below must pass that profile to start it.
NIM_COMPOSE_PROFILE = "text-embed"


def _quote_env_value(value: str) -> str:
    return '"' + str(value).replace('\\', '\\\\').replace('"', '\\"') + '"'


def write_launchable_env(updates: dict[str, str], reset: bool = False) -> None:
    """Write an idempotent launchable env block and refresh os.environ.

    The notebook can be run repeatedly in the same checkout. Keeping all values
    in one generated block prevents stale optional settings such as
    ENABLE_VLM_INFERENCE=true from carrying into a later default deployment.
    """
    global LAUNCHABLE_ENV_VALUES
    if reset:
        LAUNCHABLE_ENV_VALUES = {}

    LAUNCHABLE_ENV_VALUES.update({key: str(value) for key, value in updates.items()})
    ENV_FILE.parent.mkdir(parents=True, exist_ok=True)
    existing = ENV_FILE.read_text(encoding="utf-8") if ENV_FILE.exists() else ""
    block_pattern = re.compile(
        rf"\n?{re.escape(LAUNCHABLE_ENV_BEGIN)}.*?{re.escape(LAUNCHABLE_ENV_END)}\n?",
        flags=re.DOTALL,
    )
    cleaned = block_pattern.sub("\n", existing).rstrip()

    lines = [LAUNCHABLE_ENV_BEGIN]
    for key in sorted(LAUNCHABLE_ENV_VALUES):
        value = LAUNCHABLE_ENV_VALUES[key]
        lines.append(f"export {key}={_quote_env_value(value)}")
        os.environ[key] = value
    lines.append(LAUNCHABLE_ENV_END)

    ENV_FILE.write_text(
        (cleaned + "\n\n" if cleaned else "") + "\n".join(lines) + "\n",
        encoding="utf-8",
    )
    load_dotenv(ENV_FILE, override=True)


def refresh_launchable_env() -> None:
    """Load deploy/compose/.env with notebook-generated values taking precedence."""
    if ENV_FILE.exists():
        load_dotenv(ENV_FILE, override=True)


# =============================================================================
# DOCKER COMPOSE HELPERS
# =============================================================================

def docker_compose(yaml_file: str, action: str, args: str = "", profile: str = "") -> bool:
    """Run docker compose command.
    - Suppresses image pull noise
    - Surfaces full error output on failure
    """

    profile_arg = f"--profile {profile}" if profile else ""
    cmd = f"docker compose -f {yaml_file} {profile_arg} {action} {args}".strip()
    cmd = " ".join(cmd.split())  # Clean up extra spaces

    # Actions that may pull images implicitly
    QUIET_ACTIONS = {"pull", "up"}

    print_prefix = "Pulling images" if action == "pull" else "Running"

    if action in QUIET_ACTIONS:
        print(f"{print_prefix} from {yaml_file}...", flush=True)

        result = subprocess.run(
            cmd,
            shell=True,
            env=os.environ,
            stdout=subprocess.PIPE,    # capture
            stderr=subprocess.PIPE,    # capture
            text=True,
        )

        if result.returncode == 0:
            print(f"✅ {action} complete", flush=True)
            return True

        # ---- Failure path: show diagnostics ----
        print(f"❌ {action} failed", flush=True)

        if result.stdout:
            print("\n--- STDOUT ---")
            print(result.stdout)

        if result.stderr:
            print("\n--- STDERR ---")
            print(result.stderr)

        return False

    # ---- Non-quiet actions: stream output ----
    print(f"$ {cmd}", flush=True)
    result = subprocess.run(cmd, shell=True, env=os.environ)

    if result.returncode == 0:
        print(f"✅ {action} succeeded", flush=True)
        return True

    print(f"❌ {action} failed (exit code {result.returncode})", flush=True)
    return False

def check_containers():
    """Show all running Docker containers."""
    subprocess.run('docker ps --format "table {{.Names}}\t{{.Status}}"', shell=True)


# =============================================================================
# API HELPERS
# =============================================================================

async def api_get(url: str, params: dict = None) -> dict:
    """Make async GET request."""
    async with aiohttp.ClientSession() as session:
        async with session.get(url, params=params) as resp:
            if resp.status != 200:
                text = await resp.text()
                print(f"❌ API error {resp.status}: {text[:200]}")
                return {"error": text, "status": resp.status}
            try:
                return await resp.json()
            except:
                text = await resp.text()
                return {"error": text, "raw": True}


async def check_health(base_url: str, name: str) -> bool:
    """Check if service is healthy."""
    try:
        await api_get(f"{base_url}/v1/health")
        print(f"✅ {name} is healthy")
        return True
    except Exception as e:
        print(f"❌ {name} not responding: {e}")
        return False


# =============================================================================
# COLLECTION MANAGEMENT
# =============================================================================

async def create_collection(name: str, collection_type: str = "text") -> dict:
    """Create a new collection in the vector database."""
    url = f"{INGESTOR_BASE_URL}/v1/collections"
    params = {
        "vdb_endpoint": ELASTICSEARCH_ENDPOINT,
        "collection_type": collection_type
    }
    async with aiohttp.ClientSession() as session:
        async with session.post(url, params=params, json=[name]) as resp:
            if resp.status == 200:
                print(f"✅ Created collection: {name}")
            else:
                text = await resp.text()
                print(f"Collection response: {resp.status} - {text[:100]}")
            try:
                return await resp.json()
            except:
                return {"status": resp.status}


async def list_collections() -> list:
    """List all vector database collections."""
    result = await api_get(
        f"{INGESTOR_BASE_URL}/v1/collections",
        {"vdb_endpoint": ELASTICSEARCH_ENDPOINT}
    )
    collections = result.get("collections", [])
    print(f"Collections ({len(collections)}):")
    for c in collections:
        print(f"  • {c.get('collection_name', c)}")
    return collections


async def delete_collection(name: str):
    """Delete a collection."""
    async with aiohttp.ClientSession() as session:
        params = {"vdb_endpoint": ELASTICSEARCH_ENDPOINT, "collection_names": [name]}
        async with session.delete(
            f"{INGESTOR_BASE_URL}/v1/collections", params=params
        ) as resp:
            print(f"Deleted collection: {name}")
            return await resp.json()


# =============================================================================
# DOCUMENT MANAGEMENT
# =============================================================================

async def upload_documents(filepaths, collection: str = "multimodal_data") -> dict:
    """
    Upload document(s) to vector database in a single request.
    
    Args:
        filepaths: Single filepath (str), list of filepaths, or directory path
        collection: Target collection name (must exist before uploading)
    
    Returns:
        Response dict with task_id for tracking upload status
    
    Chunking options:
        - chunk_size: 1024 tokens
        - chunk_overlap: 150 tokens
    """
    # Handle single file, list, or directory
    if isinstance(filepaths, str):
        if os.path.isdir(filepaths):
            # It's a directory - get all files
            filepaths = [
                os.path.join(filepaths, f) 
                for f in os.listdir(filepaths) 
                if os.path.isfile(os.path.join(filepaths, f))
            ]
        else:
            filepaths = [filepaths]
    
    # Configure upload parameters
    data = {
        "vdb_endpoint": ELASTICSEARCH_ENDPOINT,
        "collection_name": collection,
        "split_options": {
            "chunk_size": 1024,
            "chunk_overlap": 150
        }
    }
    
    # Prepare multipart form data with all files
    form_data = aiohttp.FormData()
    for filepath in filepaths:
        form_data.add_field(
            "documents",
            open(filepath, "rb"),
            filename=os.path.basename(filepath),
            content_type="application/pdf"
        )
        print(f"📤 Adding: {os.path.basename(filepath)}")
    form_data.add_field("data", json.dumps(data), content_type="application/json")
    
    # Upload all documents in single request
    async with aiohttp.ClientSession() as session:
        try:
            async with session.post(
                f"{INGESTOR_BASE_URL}/v1/documents", data=form_data
            ) as resp:
                result = await resp.json()
                task_id = result.get("task_id", "")
                print(f"✅ Upload started → Task ID: {task_id}")
                return result
        except aiohttp.ClientError as e:
            print(f"❌ Upload error: {e}")
            return {"error": str(e)}


async def check_upload_status(task_id: str) -> dict:
    """Check document upload status."""
    result = await api_get(
        f"{INGESTOR_BASE_URL}/v1/status",
        {"task_id": task_id}
    )
    print(f"Task {task_id[:8]}...: {result.get('state', 'UNKNOWN')}")
    return result


async def list_documents(collection: str = "multimodal_data") -> list:
    """List documents in collection."""
    result = await api_get(
        f"{INGESTOR_BASE_URL}/v1/documents",
        {"collection_name": collection, "vdb_endpoint": ELASTICSEARCH_ENDPOINT}
    )
    docs = result.get("documents", [])
    print(f"Documents in '{collection}' ({len(docs)}):")
    for d in docs:
        print(f"  • {d.get('document_name', d)}")
    return docs


async def delete_document(filenames, collection: str = "multimodal_data"):
    """Delete document(s) from collection. Accepts single filename or list."""
    if isinstance(filenames, str):
        filenames = [filenames]
    
    async with aiohttp.ClientSession() as session:
        params = {"collection_name": collection, "vdb_endpoint": ELASTICSEARCH_ENDPOINT}
        async with session.delete(
            f"{INGESTOR_BASE_URL}/v1/documents",
            params=params,
            json=filenames
        ) as resp:
            for f in filenames:
                print(f"🗑️  Deleted: {f}")
            return await resp.json()


# =============================================================================
# CHAT / QUERY
# =============================================================================

async def chat(question: str, use_rag: bool = True, collection: str = "multimodal_data", 
               show_citations: bool = True) -> str:
    """
    Ask a question to the RAG system.
    
    Args:
        question: Your question
        use_rag: If True, search documents first (default). If False, use LLM only.
        collection: Collection to search
        show_citations: If True, print sources used to generate the response
    
    Returns:
        The AI response text
    """
    payload = {
        "messages": [{"role": "user", "content": question}],
        "use_knowledge_base": use_rag,
        "collection_names": [collection] if use_rag else [],
        "temperature": 0.2,
        "model": "nvidia/nemotron-3-super-120b-a12b"
    }

    async with aiohttp.ClientSession() as session:
        async with session.post(
            f"{RAG_BASE_URL}/v1/chat/completions", json=payload
        ) as resp:
            content_type = resp.headers.get("Content-Type", "")
            
            if "text/event-stream" in content_type:
                # Handle streaming response - read full response first
                response_text = await resp.text()
                concatenated_content = ""
                citations = None
                
                for line in response_text.split("\n"):
                    if line.startswith("data: "):
                        json_str = line[6:]
                        if json_str.strip() == "[DONE]":
                            continue
                        try:
                            data = json.loads(json_str)
                            content = data.get("choices", [{}])[0].get("delta", {}).get("content", "")
                            concatenated_content += content
                            # Citations are in the first chunk
                            if citations is None and "citations" in data:
                                citations = data.get("citations", {})
                        except json.JSONDecodeError:
                            continue
                
                print(concatenated_content)
                
                # Print citations if available and requested
                if show_citations and use_rag and citations:
                    results = citations.get("results", [])
                    if results:
                        print("\n" + "-" * 60)
                        print(f"📚 SOURCES ({len(results)} citations):")
                        print("-" * 60)
                        for i, src in enumerate(results, 1):
                            doc_name = src.get("document_name", "unknown")
                            doc_type = src.get("document_type", "text")
                            score = src.get("score", 0)
                            content = src.get("content", "")
                            print(f"[{i}] {doc_name} ({doc_type}) - Score: {score:.4f}")
                            # Display base64 images inline
                            if content.startswith(("/9j/", "iVBOR")):
                                try:
                                    img_data = base64.b64decode(content)
                                    display(IPImage(data=img_data, width=400))
                                except Exception as e:
                                    print(f"    [Image decode error: {e}]")
                            elif content.startswith("data:image"):
                                try:
                                    img_b64 = content.split(",", 1)[1]
                                    img_data = base64.b64decode(img_b64)
                                    display(IPImage(data=img_data, width=400))
                                except Exception as e:
                                    print(f"    [Image decode error: {e}]")
                            else:
                                content_preview = content[:150].replace("\n", " ")
                                print(f"    {content_preview}...")
                
                return concatenated_content
            else:
                # Handle regular JSON response
                response_json = await resp.json()
                if "error" in response_json:
                    print(f"Error: {response_json['error']}")
                    return ""
                
                content = response_json.get("choices", [{}])[0].get("message", {}).get("content", "")
                #print(content)
                
                # Print citations if available
                if show_citations and use_rag:
                    citations = response_json.get("citations", {})
                    results = citations.get("results", [])
                    if results:
                        print("\n" + "-" * 60)
                        print(f"📚 SOURCES ({len(results)} citations):")
                        print("-" * 60)
                        for i, src in enumerate(results, 1):
                            doc_name = src.get("document_name", "unknown")
                            doc_type = src.get("document_type", "text")
                            score = src.get("score", 0)
                            content = src.get("content", "")
                            print(f"[{i}] {doc_name} ({doc_type}) - Score: {score:.4f}")
                            # Display base64 images inline
                            if content.startswith(("/9j/", "iVBOR")):
                                try:
                                    img_data = base64.b64decode(content)
                                    display(IPImage(data=img_data, width=400))
                                except Exception as e:
                                    print(f"    [Image decode error: {e}]")
                            elif content.startswith("data:image"):
                                try:
                                    img_b64 = content.split(",", 1)[1]
                                    img_data = base64.b64decode(img_b64)
                                    display(IPImage(data=img_data, width=400))
                                except Exception as e:
                                    print(f"    [Image decode error: {e}]")
                            else:
                                content_preview = content[:150].replace("\n", " ")
                                print(f"    {content_preview}...")
                
                return content


# =============================================================================
# DEPLOYMENT COMMANDS
# =============================================================================

def deploy_all():
    """Start all RAG Blueprint services."""
    if globals().get("SYSTEM_REQUIREMENTS_OK") is False:
        raise RuntimeError(
            "System requirements check failed. Fix the reported errors before deploying."
        )

    refresh_launchable_env()

    print("=" * 60)
    print("DEPLOYING NVIDIA RAG BLUEPRINT")
    print("=" * 60)
   
    print("\n[1/4] NIM Microservices...")
    print(f"Starting launchable NIM services: {NIM_SERVICE_ARGS}")
    docker_compose(
        "deploy/compose/nims.yaml",
        "pull",
        f"-q {NIM_SERVICE_ARGS}",
        profile=NIM_COMPOSE_PROFILE,
    )
    docker_compose(
        "deploy/compose/nims.yaml",
        "up",
        f"-d {NIM_SERVICE_ARGS}",
        profile=NIM_COMPOSE_PROFILE,
    )
    print("-" * 60)
    print("\n[2/4] Vector Database...")
    docker_compose("deploy/compose/vectordb.yaml", "pull", "-q")
    docker_compose("deploy/compose/vectordb.yaml", "up", "-d")
    print("-" * 60)
    print("\n[3/4] Ingestor Server...")
    docker_compose("deploy/compose/docker-compose-ingestor-server.yaml", "pull", "-q")
    docker_compose("deploy/compose/docker-compose-ingestor-server.yaml", "up", "-d")
    print("-" * 60)
    print("\n[4/4] RAG Server...")
    docker_compose("deploy/compose/docker-compose-rag-server.yaml", "pull", "-q")
    docker_compose("deploy/compose/docker-compose-rag-server.yaml", "up", "-d")
    
    print("\n" + "=" * 60)
    print("DEPLOYMENT COMPLETE")
    print("Wait 2-5 minutes for services to become healthy.")
    print("=" * 60)


def stop_all():
    """Stop all RAG Blueprint services."""
    print("=" * 60)
    print("STOPPING ALL SERVICES")
    print("=" * 60)
    
    print("\n[1/4] Stopping RAG Server...")
    docker_compose("deploy/compose/docker-compose-rag-server.yaml", "down")
    
    print("\n[2/4] Stopping Ingestor Server...")
    docker_compose("deploy/compose/docker-compose-ingestor-server.yaml", "down")
    
    print("\n[3/4] Stopping Vector Database...")
    docker_compose("deploy/compose/vectordb.yaml", "down")
    
    print("\n[4/4] Stopping NIM Containers...")
    docker_compose("deploy/compose/nims.yaml", "down", profile=NIM_COMPOSE_PROFILE)
    
    print("\n" + "=" * 60)
    print("✅ ALL SERVICES STOPPED")
    print("=" * 60)


# =============================================================================
# INITIALIZATION MESSAGE
# =============================================================================

print("✅ Helper functions loaded!")
print()
print("Available functions:")
print("  deploy_all()            - Start all services")
print("  stop_all()              - Stop all services")
print("  check_containers()      - List running containers")
print("  chat(question)          - Ask a question (use_rag=True by default)")
print("  upload_documents(paths) - Upload file(s) or directory")
print("  list_documents()        - List uploaded files")

---

# Section 1: Check Prerequisites

---

Verify your system meets the requirements before deploying.

### 1.1 System Requirements Check

In [ ]:
import os
import subprocess
import re
import shutil
from pathlib import Path

MIN_SYSTEM_MEMORY_GIB = 32
LAUNCHABLE_NIM_SHM_GIB_BY_SERVICE = {
    "nemotron-embedding-ms": 16,
    "nemotron-ranking-ms": 16,
    "page-elements": 16,
    "graphic-elements": 16,
    "table-structure": 16,
    "nemotron-ocr": 16,
}
LAUNCHABLE_NIM_SHM_GIB = sum(LAUNCHABLE_NIM_SHM_GIB_BY_SERVICE.values())
MIN_DOCKER_MEMORY_OVERHEAD_GIB = 16
MIN_DOCKER_MEMORY_GIB = LAUNCHABLE_NIM_SHM_GIB + MIN_DOCKER_MEMORY_OVERHEAD_GIB
MIN_GPU_VRAM_GIB = 75
MIN_TOTAL_DISK_FREE_GIB = 200
MIN_DOCKER_IMAGE_DISK_GIB = 150
MIN_MODEL_CACHE_DISK_GIB = 150


def bytes_to_gib(value: int) -> float:
    return value / (1024 ** 3)


def mib_to_gib(value: int) -> float:
    return value / 1024


def existing_path(path: Path) -> Path:
    path = path.expanduser()
    while not path.exists() and path != path.parent:
        path = path.parent
    return path


def free_disk_gib(path: Path) -> float:
    return bytes_to_gib(shutil.disk_usage(existing_path(path)).free)

def print_docker_relocation_steps():
    """Print manual steps for moving Docker's data-root to a larger disk."""
    print("")
    print("    ─────────────────────────────────────────────────────────────────")
    print("    📦 HOW TO RELOCATE DOCKER'S DATA-ROOT TO A LARGER DISK")
    print("    ─────────────────────────────────────────────────────────────────")
    print("      1. Identify a mount with sufficient free space:")
    print("             df -h")
    print("      2. Stop Docker and containerd:")
    print("             sudo systemctl stop docker docker.socket containerd")
    print("      3. Add the new data-root to /etc/docker/daemon.json (preserves existing keys):")
    print("             sudo mkdir -p /ephemeral/docker")
    print("             sudo python3 -c \"import json, pathlib; p=pathlib.Path('/etc/docker/daemon.json'); d=json.loads(p.read_text()) if p.exists() and p.read_text().strip() else {}; d['data-root']='/ephemeral/docker'; p.parent.mkdir(parents=True, exist_ok=True); p.write_text(json.dumps(d, indent=2))\"")
    print("      4. Move existing data (skip if /var/lib/docker is empty):")
    print("             sudo rsync -a /var/lib/docker/ /ephemeral/docker/")
    print("             sudo mv /var/lib/docker /var/lib/docker.old")
    print("      5. Restart services and verify the new data-root:")
    print("             sudo systemctl start containerd docker")
    print("             docker info | grep 'Docker Root Dir'")
    print("      After confirming images pull correctly, reclaim old space:")
    print("             sudo rm -rf /var/lib/docker.old")
    print("    ─────────────────────────────────────────────────────────────────")



def same_filesystem(first: Path, second: Path) -> bool:
    return os.stat(existing_path(first)).st_dev == os.stat(existing_path(second)).st_dev


print("=" * 70)
print("SYSTEM REQUIREMENTS CHECK")
print("=" * 70)

errors = []
warnings = []

# ─────────────────────────────────────────────────────────────────────────────
# [1] Operating System
# ─────────────────────────────────────────────────────────────────────────────
print("\n[1] Operating System (Ubuntu 22.04+ recommended):")
try:
    result = subprocess.run(["lsb_release", "-d"], capture_output=True, text=True)
    if result.returncode == 0:
        os_info = result.stdout.strip()
        print(f"    {os_info}")
        if "Ubuntu" in os_info:
            # Extract version number
            match = re.search(r"(\d+\.\d+)", os_info)
            if match:
                version = float(match.group(1))
                if version >= 22.04:
                    print("    ✅ PASS")
                else:
                    warnings.append(f"Ubuntu {version} detected, 22.04+ recommended")
                    print(f"    ⚠️  WARNING: Ubuntu {version} < 22.04")
        else:
            warnings.append("Non-Ubuntu OS detected")
            print("    ⚠️  WARNING: Non-Ubuntu OS")
    else:
        print("    Unable to determine OS")
except FileNotFoundError:
    print("    Non-Linux system (macOS/Windows)")
    warnings.append("Non-Linux system detected")

# ─────────────────────────────────────────────────────────────────────────────
# [2] NVIDIA GPU and VRAM
# ─────────────────────────────────────────────────────────────────────────────
print("\n[2] NVIDIA GPU and VRAM:")
try:
    result = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
    if result.returncode == 0 and result.stdout.strip():
        gpus = result.stdout.strip().split("\n")
        for gpu in gpus:
            print(f"    {gpu}")
        print(f"    ✅ PASS ({len(gpus)} GPU(s) detected)")

        vram_result = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=name,memory.total",
                "--format=csv,noheader,nounits",
            ],
            capture_output=True,
            text=True,
        )
        if vram_result.returncode == 0 and vram_result.stdout.strip():
            total_vram_mib = 0
            for line in vram_result.stdout.strip().splitlines():
                name, memory_mib = line.rsplit(",", 1)
                memory_mib = int(memory_mib.strip())
                total_vram_mib += memory_mib
                print(f"    {name.strip()}: {mib_to_gib(memory_mib):.1f} GiB VRAM")
            total_vram_gib = mib_to_gib(total_vram_mib)
            if total_vram_gib >= MIN_GPU_VRAM_GIB:
                print(f"    ✅ PASS ({total_vram_gib:.1f} GiB total VRAM)")
            else:
                errors.append(
                    f"Only {total_vram_gib:.1f} GiB total GPU VRAM; "
                    f"need {MIN_GPU_VRAM_GIB}+ GiB"
                )
                print(
                    f"    ❌ FAIL: {total_vram_gib:.1f} GiB < "
                    f"{MIN_GPU_VRAM_GIB} GiB required"
                )
        else:
            warnings.append("Unable to determine GPU VRAM")
            print("    ⚠️  WARNING: Unable to determine GPU VRAM")
    else:
        errors.append("No NVIDIA GPU detected")
        print("    ❌ FAIL: No NVIDIA GPU detected")
except FileNotFoundError:
    errors.append("nvidia-smi not found - NVIDIA driver not installed")
    print("    ❌ FAIL: nvidia-smi not found")

# ─────────────────────────────────────────────────────────────────────────────
# [3] NVIDIA Driver Version (need 560+)
# ─────────────────────────────────────────────────────────────────────────────
print("\n[3] NVIDIA Driver Version (need 560+):")
try:
    result = subprocess.run(["nvidia-smi", "-q"], capture_output=True, text=True)
    if result.returncode == 0:
        match = re.search(r"Driver Version\s*:\s*(\d+)", result.stdout)
        if match:
            driver_version = int(match.group(1))
            print(f"    Driver Version: {driver_version}")
            if driver_version >= 560:
                print("    ✅ PASS")
            else:
                errors.append(f"Driver version {driver_version} < 560 required")
                print(f"    ❌ FAIL: Version {driver_version} < 560")
        else:
            print("    Unable to parse driver version")
except FileNotFoundError:
    print("    ❌ FAIL: nvidia-smi not found")

# ─────────────────────────────────────────────────────────────────────────────
# [4] CUDA Version (need 12.4+)
# ─────────────────────────────────────────────────────────────────────────────
print("\n[4] CUDA Version (need 12.4+):")
try:
    result = subprocess.run(["nvidia-smi", "-q"], capture_output=True, text=True)
    if result.returncode == 0:
        match = re.search(r"CUDA Version\s*:\s*(\d+\.\d+)", result.stdout)
        if match:
            cuda_version = float(match.group(1))
            print(f"    CUDA Version: {cuda_version}")
            if cuda_version >= 12.4:
                print("    ✅ PASS")
            else:
                errors.append(f"CUDA version {cuda_version} < 12.4 required")
                print(f"    ❌ FAIL: Version {cuda_version} < 12.4")
        else:
            print("    Unable to parse CUDA version")
except FileNotFoundError:
    print("    ❌ FAIL: nvidia-smi not found")

# ─────────────────────────────────────────────────────────────────────────────
# [5] Docker Version (need 26.0+)
# ─────────────────────────────────────────────────────────────────────────────
print("\n[5] Docker Version (need 26.0+):")
if shutil.which("docker"):
    result = subprocess.run(["docker", "--version"], capture_output=True, text=True)
    if result.returncode == 0:
        print(f"    {result.stdout.strip()}")
        match = re.search(r"(\d+)\.\d+", result.stdout)
        if match:
            docker_major = int(match.group(1))
            if docker_major >= 26:
                print("    ✅ PASS")
            else:
                errors.append(f"Docker version {docker_major} < 26 required")
                print(f"    ❌ FAIL: Major version {docker_major} < 26")
else:
    errors.append("Docker not installed")
    print("    ❌ FAIL: Docker not installed")

# ─────────────────────────────────────────────────────────────────────────────
# [6] Docker Compose Version (need 2.29.1+)
# ─────────────────────────────────────────────────────────────────────────────
print("\n[6] Docker Compose Version (need 2.29.1+):")
if shutil.which("docker"):
    result = subprocess.run(["docker", "compose", "version"], capture_output=True, text=True)
    if result.returncode == 0:
        print(f"    {result.stdout.strip()}")
        match = re.search(r"v?(\d+)\.(\d+)", result.stdout)
        if match:
            major, minor = int(match.group(1)), int(match.group(2))
            if major > 2 or (major == 2 and minor >= 29):
                print("    ✅ PASS")
            else:
                errors.append(f"Docker Compose {major}.{minor} < 2.29.1 required")
                print(f"    ❌ FAIL: Version {major}.{minor} < 2.29.1")
    else:
        errors.append("Docker Compose not available")
        print("    ❌ FAIL: Docker Compose not available")
else:
    print("    ❌ FAIL: Docker not installed")

# ─────────────────────────────────────────────────────────────────────────────
# [7] Host System Memory (need 32 GiB+)
# ─────────────────────────────────────────────────────────────────────────────
print("\n[7] Host System Memory (need 32 GiB+):")
try:
    meminfo = {}
    with open("/proc/meminfo", "r", encoding="utf-8") as file:
        for line in file:
            key, value = line.split(":", 1)
            meminfo[key] = int(value.strip().split()[0]) * 1024
    total_gib = bytes_to_gib(meminfo.get("MemTotal", 0))
    available_gib = bytes_to_gib(meminfo.get("MemAvailable", 0))
    print(f"    Total: {total_gib:.1f} GiB")
    print(f"    Available: {available_gib:.1f} GiB")
    if total_gib >= MIN_SYSTEM_MEMORY_GIB:
        print(f"    ✅ PASS ({total_gib:.1f} GiB total)")
    elif total_gib >= 16:
        warnings.append(f"Only {total_gib:.1f} GiB host RAM, 32 GiB+ recommended")
        print(f"    ⚠️  WARNING: {total_gib:.1f} GiB < 32 GiB recommended")
    else:
        errors.append(
            f"Only {total_gib:.1f} GiB host RAM; "
            f"need {MIN_SYSTEM_MEMORY_GIB}+ GiB for deployment"
        )
        print(f"    ❌ FAIL: {total_gib:.1f} GiB < 32 GiB minimum")
except FileNotFoundError:
    print("    Unable to check (non-Linux system)")

# ─────────────────────────────────────────────────────────────────────────────
# [8] Docker Daemon Memory
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n[8] Docker Daemon Memory (need {MIN_DOCKER_MEMORY_GIB} GiB+):")
print(f"    Launchable NIM shared-memory configuration: {LAUNCHABLE_NIM_SHM_GIB} GiB")
print(f"    Added overhead budget: {MIN_DOCKER_MEMORY_OVERHEAD_GIB} GiB")
if shutil.which("docker"):
    result = subprocess.run(
        ["docker", "info", "--format", "{{.MemTotal}}"],
        capture_output=True,
        text=True,
    )
    docker_mem_text = result.stdout.strip()
    if result.returncode == 0 and docker_mem_text.isdigit():
        docker_memory_gib = bytes_to_gib(int(docker_mem_text))
        print(f"    Docker daemon memory: {docker_memory_gib:.1f} GiB")
        if docker_memory_gib >= MIN_DOCKER_MEMORY_GIB:
            print(f"    ✅ PASS ({docker_memory_gib:.1f} GiB available to Docker)")
        else:
            errors.append(
                f"Docker daemon has only {docker_memory_gib:.1f} GiB memory; "
                f"increase Docker Desktop/daemon memory to {MIN_DOCKER_MEMORY_GIB}+ GiB "
                "or reduce the local NIM service set"
            )
            print(
                f"    ❌ FAIL: Docker daemon memory {docker_memory_gib:.1f} GiB < "
                f"{MIN_DOCKER_MEMORY_GIB} GiB required"
            )
    else:
        errors.append("Docker daemon is unavailable or did not report memory")
        print("    ❌ FAIL: Docker daemon unavailable or memory unknown")
else:
    print("    ❌ FAIL: Docker not installed")

# ─────────────────────────────────────────────────────────────────────────────
# [9] Disk Space for Docker Images and Model Cache
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n[9] Disk Space (need {MIN_TOTAL_DISK_FREE_GIB} GiB+ total free):")
docker_root = None
if shutil.which("docker"):
    result = subprocess.run(
        ["docker", "info", "--format", "{{.DockerRootDir}}"],
        capture_output=True,
        text=True,
    )
    if result.returncode == 0 and result.stdout.strip():
        docker_root = Path(result.stdout.strip())
        print(f"    Docker root: {docker_root}")
    else:
        errors.append("Docker root directory unavailable; cannot verify image-pull disk space")
        print("    ❌ FAIL: Docker root directory unavailable")
else:
    print("    ❌ FAIL: Docker not installed")

model_cache_dir = Path(os.environ.get("MODEL_DIRECTORY", "~/.cache/model-cache")).expanduser()
model_cache_dir.mkdir(parents=True, exist_ok=True)
print(f"    Model cache: {model_cache_dir}")

if docker_root is not None:
    docker_free_gib = free_disk_gib(docker_root)
    model_cache_free_gib = free_disk_gib(model_cache_dir)
    print(f"    Docker filesystem free: {docker_free_gib:.1f} GiB")
    print(f"    Model-cache filesystem free: {model_cache_free_gib:.1f} GiB")
    if same_filesystem(docker_root, model_cache_dir):
        if docker_free_gib >= MIN_TOTAL_DISK_FREE_GIB:
            print(f"    ✅ PASS ({docker_free_gib:.1f} GiB free for images + model cache)")
        else:
            errors.append(
                f"Only {docker_free_gib:.1f} GiB free on the shared Docker/model-cache filesystem; "
                f"need {MIN_TOTAL_DISK_FREE_GIB}+ GiB before pulling images and caching models"
            )
            print(
                f"    ❌ FAIL: {docker_free_gib:.1f} GiB < "
                f"{MIN_TOTAL_DISK_FREE_GIB} GiB required"
            )
            print_docker_relocation_steps()
    else:
        disk_ok = True
        if docker_free_gib < MIN_DOCKER_IMAGE_DISK_GIB:
            disk_ok = False
            errors.append(
                f"Only {docker_free_gib:.1f} GiB free for Docker images; "
                f"need {MIN_DOCKER_IMAGE_DISK_GIB}+ GiB"
            )
            print(
                f"    ❌ FAIL: Docker image space {docker_free_gib:.1f} GiB < "
                f"{MIN_DOCKER_IMAGE_DISK_GIB} GiB required"
            )
            print_docker_relocation_steps()
        if model_cache_free_gib < MIN_MODEL_CACHE_DISK_GIB:
            disk_ok = False
            errors.append(
                f"Only {model_cache_free_gib:.1f} GiB free for NIM model cache; "
                f"need {MIN_MODEL_CACHE_DISK_GIB}+ GiB"
            )
            print(
                f"    ❌ FAIL: model-cache space {model_cache_free_gib:.1f} GiB < "
                f"{MIN_MODEL_CACHE_DISK_GIB} GiB required"
            )
        if disk_ok:
            print("    ✅ PASS (separate Docker and model-cache filesystems have enough space)")

# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)

SYSTEM_REQUIREMENTS_OK = not errors

if errors:
    print("\n" + "🚨" * 35)
    print("🚨" + " " * 66 + "🚨")
    print("🚨   REQUIREMENTS NOT MET - CANNOT PROCEED WITH DEPLOYMENT!        🚨")
    print("🚨" + " " * 66 + "🚨")
    print("🚨" * 35)
    print("\n❌ ERRORS ({} issues):".format(len(errors)))
    for err in errors:
        print(f"   • {err}")
    if warnings:
        print("\n⚠️  WARNINGS ({} issues):".format(len(warnings)))
        for warn in warnings:
            print(f"   • {warn}")
    print("\n" + "=" * 70)
    print("Please fix the above errors before continuing.")
    print("See: https://docs.nvidia.com/rag/latest/support-matrix.html")
    print("=" * 70)
    raise RuntimeError("System requirements not met; fix the errors above before deploying.")
elif warnings:
    print("\n✅ REQUIREMENTS MET (with warnings)")
    print("\n⚠️  WARNINGS ({} issues):".format(len(warnings)))
    for warn in warnings:
        print(f"   • {warn}")
    print("\nYou may proceed, but some features might not work as expected.")
    print("=" * 70)
else:
    print("    ✅   ALL REQUIREMENTS MET - READY TO DEPLOY!")

---

# Section 2: Deploy the Blueprint

---

This section will guide you through:
1. Setting your NGC API key for authentication
2. Cloning the NVIDIA RAG Blueprint repository  
3. Logging into NGC and configuring the environment
4. Deploying all services with Docker Compose
5. Verifying the deployment status

### 2.1 Set Your NGC API Key

Get your key at: https://org.ngc.nvidia.com/setup/api-keys

Your key must have permissions for

* NGC Catalog
* Public API Endpoints

> **Note**: Cell 2.1a below is optional - only run it if you need to clear a previously set API key.

### 2.1a Reset API Key (Optional)

⚠️ **Only run this cell if you need to clear a previously set API key and start over.**

In [ ]:
import os

os.environ.pop("NGC_API_KEY", None)
print("✅ NGC_API_KEY unset (if it existed)")

**Enter Your NGC API Key:**

In [ ]:
import getpass
import os

if os.environ.get("NGC_API_KEY", "").startswith("nvapi-"):
    print("✅ NGC_API_KEY already set")
else:
    key = getpass.getpass("Enter NGC API key (starts with 'nvapi-'): ")
    if key.startswith("nvapi-"):
        os.environ["NGC_API_KEY"] = key
        print("✅ API key set")
    else:
        print("❌ Invalid key format - must start with 'nvapi-'")

### 2.2 Clone the GitHub Repository

Clone the official NVIDIA RAG Blueprint repository and checkout the v2.4 release:

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/NVIDIA-AI-Blueprints/rag.git"
#BRANCH = "release-v2.5.0"
BRANCH = "develop"
# Check if we're already in the rag repo (look for deploy/compose)
if os.path.exists("deploy/compose"):
    print(f"✅ Already in rag repo: {os.getcwd()}")
elif os.path.exists("../deploy/compose"):
    # We're in a subdirectory (like notebooks/), go up
    os.chdir("..")
    print(f"✅ Changed to repo root: {os.getcwd()}")
else:
    # Clone if rag folder doesn't exist
    if not os.path.exists("rag"):
        print(f"Cloning {REPO_URL}...")
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL], check=True)
        print("✅ Clone complete")
    os.chdir("rag")
    print(f"✅ Changed to: {os.getcwd()}")

### 2.3 Login to NGC & Configure Environment

In [ ]:
import subprocess

# Login to NGC container registry
!echo "${NGC_API_KEY}" | docker login nvcr.io -u '$oauthtoken' --password-stdin

# Set user ID for Docker volume permissions
user_id = subprocess.check_output("id -u", shell=True).decode().strip()
os.environ["USERID"] = user_id

# Create model cache directory
model_cache = os.path.expanduser("~/.cache/model-cache")
os.makedirs(model_cache, exist_ok=True)
os.environ["MODEL_DIRECTORY"] = model_cache

# Configure the launchable default path to use NVIDIA-hosted LLM endpoints.
# Reset optional feature toggles here so re-running the notebook starts from the
# standard LLM pipeline unless the optional VLM/reasoning cells are run again.
base_env = {
    "USERID": user_id,
    "MODEL_DIRECTORY": model_cache,
    "PROMPT_CONFIG_FILE": str(Path("src/nvidia_rag/rag_server/prompt.yaml").resolve()),
    "APP_LLM_SERVERURL": "",
    "SUMMARY_LLM_SERVERURL": "",
    "SUMMARY_LLM": "nvidia/nemotron-3-super-120b-a12b",
    "APP_LLM_MODELNAME": "nvidia/nemotron-3-super-120b-a12b",
    "APP_FILTEREXPRESSIONGENERATOR_MODELNAME": "nvidia/nemotron-3-super-120b-a12b",
    "APP_FILTEREXPRESSIONGENERATOR_SERVERURL": "",
    "APP_QUERYREWRITER_MODELNAME": "nvidia/nemotron-3-super-120b-a12b",
    "APP_QUERYREWRITER_SERVERURL": "",
    # Use the text embedding model (overrides the VLM embedding default in nvdev.env).
    "APP_EMBEDDINGS_MODELNAME": "nvidia/llama-nemotron-embed-1b-v2",
    "APP_EMBEDDINGS_SERVERURL": "nemotron-embedding-ms:8000/v1",
    "ENABLE_VLM_INFERENCE": "false",
    "VLM_TO_LLM_FALLBACK": "true",
    "APP_NVINGEST_EXTRACTIMAGES": "False",
    "LLM_ENABLE_THINKING": "false",
    "LLM_REASONING_BUDGET": "0",
    "LLM_LOW_EFFORT": "false",
    "FILTER_THINK_TOKENS": "true",
    "APP_VLM_ENABLE_THINKING": "false",
    "APP_VLM_THINKING_TOKEN_BUDGET": "0",
    "VLM_FILTER_THINK_TOKENS": "true",
}

write_launchable_env(base_env, reset=True)

print("✅ Base launchable configuration written")
print("   Pipeline: standard LLM generation (VLM inference disabled)")
print("   Embeddings: text model (nvidia/llama-nemotron-embed-1b-v2)")
print(f"   Model cache: {model_cache}")

### 2.3a Save Extracted Content (Optional)

⚠️ **Only run this cell if you want to save extracted document content to disk.**

This enables saving the extracted text/images from documents during ingestion for inspection or debugging.

In [ ]:
# Enable saving extracted content to disk.
# Results are written into the `rag-vol-ingestor` Docker named volume
# (host path: /var/lib/docker/volumes/rag-vol-ingestor/_data/).
save_config = {
    "APP_NVINGEST_SAVETODISK": "True",
    # Container internal path (customize as needed)
    "INGESTOR_SERVER_DATA_DIR": "/data/",
}

write_launchable_env(save_config)

print("✅ Save-to-disk configuration written")
print("   Extracted content will be saved inside the `rag-vol-ingestor` Docker volume.")


### 2.3b Enable LLM Reasoning (Optional)

⚠️ **Only run this cell if you want to enable LLM reasoning through `LLM_ENABLE_THINKING`.**


In [ ]:
# Optional: enable LLM reasoning for Nemotron 3 models.
# Current RAG flow uses environment variables, not prompt directive edits.
# To disable reasoning again, re-run cell 2.3 (base configuration).

llm_reasoning_config = {
    "LLM_ENABLE_THINKING": "true",
    "LLM_REASONING_BUDGET": "256",
    "LLM_LOW_EFFORT": "true",
    "FILTER_THINK_TOKENS": "true",
    "LLM_TEMPERATURE": "0.6",
    "LLM_TOP_P": "0.95",
}

write_launchable_env(llm_reasoning_config)

print("✅ LLM reasoning enabled")
print("   LLM_ENABLE_THINKING=true")
print("   Reasoning budget: 256 tokens, low effort mode enabled")


### 2.3c Image Captioning & VLM Inferencing (Optional)

⚠️ **Only run this cell if you want to enable image extraction, captioning, and VLM-based answering.**

This enables:

- **Image extraction** from documents during ingestion
- **Image captioning** using a Vision-Language Model (VLM)
- **VLM inference** for multimodal queries at runtime

Text-only prompts continue to fall back to the LLM unless you disable `VLM_TO_LLM_FALLBACK`.


In [ ]:
# Enable image captioning during ingestion and VLM inferencing for multimodal queries.
# This switches the RAG server to the VLM-capable path. Text-only prompts still
# fall back to the LLM because VLM_TO_LLM_FALLBACK remains true.
captioning_and_vlm_config = {
    "APP_NVINGEST_EXTRACTIMAGES": "True",
    "APP_NVINGEST_CAPTIONENDPOINTURL": "https://integrate.api.nvidia.com/v1/chat/completions",
    "APP_NVINGEST_CAPTIONMODELNAME": "nvidia/nemotron-nano-12b-v2-vl",
    "ENABLE_VLM_INFERENCE": "true",
    "VLM_TO_LLM_FALLBACK": "true",
    "APP_VLM_MODELNAME": "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning",
    "APP_VLM_SERVERURL": "https://integrate.api.nvidia.com/v1/",
    "APP_VLM_TEMPERATURE": "0.3",
    "APP_VLM_TOP_P": "0.91",
    "APP_VLM_MAX_TOKENS": "8192",
}

write_launchable_env(captioning_and_vlm_config)

print("✅ Image captioning & VLM configuration written")
print("   VLM inference: enabled with LLM fallback for text-only requests")
print("   Caption model: nvidia/nemotron-nano-12b-v2-vl")


### 2.3d Enable VLM Reasoning (Optional)

⚠️ **Only run this cell if you enabled VLM inference above and also want VLM reasoning.**


In [ ]:
# Optional: enable VLM reasoning for Nemotron Omni.
# Current VLM flow uses APP_VLM_ENABLE_THINKING and APP_VLM_THINKING_TOKEN_BUDGET.
# To disable VLM reasoning again, re-run cell 2.3 (base configuration), then
# re-run cell 2.3c if you still want VLM inference without VLM reasoning.

vlm_reasoning_config = {
    "APP_VLM_ENABLE_THINKING": "true",
    "APP_VLM_THINKING_TOKEN_BUDGET": "8192",
    "VLM_FILTER_THINK_TOKENS": "true",
}

write_launchable_env(vlm_reasoning_config)

print("✅ VLM reasoning enabled")
print("   APP_VLM_ENABLE_THINKING=true")
print("   VLM thinking token budget: 8192")


### 2.3e Load Environment Configuration

**Run this cell after configuring the optional features above to confirm the active pipeline and thinking settings.**


In [ ]:
# Check active launchable generation and reasoning settings.
refresh_launchable_env()


def env_bool(name: str, default: str = "false") -> bool:
    return os.environ.get(name, default).strip().lower() == "true"


vlm_enabled = env_bool("ENABLE_VLM_INFERENCE")
llm_thinking_enabled = env_bool("LLM_ENABLE_THINKING")
vlm_thinking_enabled = env_bool("APP_VLM_ENABLE_THINKING")
image_extraction_enabled = env_bool("APP_NVINGEST_EXTRACTIMAGES")

if vlm_enabled:
    print("\n   🔮 VLM inference: ENABLED")
    print(f"      ├─ VLM model: {os.environ.get('APP_VLM_MODELNAME', 'not set')}")
    print(f"      ├─ VLM endpoint: {os.environ.get('APP_VLM_SERVERURL', 'not set')}")
    print(
        "      ├─ Text-only fallback to LLM: "
        f"{'ENABLED' if env_bool('VLM_TO_LLM_FALLBACK', 'true') else 'DISABLED'}"
    )
    if image_extraction_enabled:
        print("      ├─ Image extraction: ENABLED")
        caption_model = os.environ.get("APP_NVINGEST_CAPTIONMODELNAME", "not set")
        print("      ├─ Image captioning: ENABLED")
        print(f"      │     Model: {caption_model}")
    else:
        print("      ├─ Image extraction: DISABLED")
        print("      ├─ Image captioning: DISABLED")
    print(
        "      └─ VLM reasoning: "
        f"{'ENABLED' if vlm_thinking_enabled else 'DISABLED'} "
        f"(APP_VLM_ENABLE_THINKING={str(vlm_thinking_enabled).lower()})"
    )
else:
    print("\n   💬 Standard LLM inference: ENABLED")
    print("      ├─ VLM inference: DISABLED")
    print(f"      ├─ LLM model: {os.environ.get('APP_LLM_MODELNAME', 'not set')}")
    print(
        "      └─ LLM reasoning: "
        f"{'ENABLED' if llm_thinking_enabled else 'DISABLED'} "
        f"(LLM_ENABLE_THINKING={str(llm_thinking_enabled).lower()})"
    )


### 2.4 Deploy All Services

**⏱️ First run takes 10-15 minutes to pull container images.**

In [ ]:
deploy_all()

### 2.5 Verify Deployment

Wait for containers to show **"healthy"** status (~2-5 minutes after deployment).

**Expected Output:**

When all services are running correctly, you should see these containers:

```
CONTAINER ID   NAMES                            STATUS
88181d20ba30   rag-frontend                     Up 2 minutes
5cf93ea91d4e   rag-server                       Up 2 minutes
03ff43bd4f53   compose-nv-ingest-ms-runtime-1   Up 2 minutes (healthy)
fcc703631b72   rag-elasticsearch                Up 3 minutes (healthy)
fcc703631b71   ingestor-server                  Up 2 minutes
77f64a4a5146   compose-redis-1                  Up 2 minutes
fe2751bfa734   nemotron-ranking-ms              Up 4 seconds (healthy)
7b5ddabf8be7   compose-graphic-elements-1       Up 10 minutes
ecfaa5190302   compose-page-elements-1          Up 10 minutes
ea8c7fdf20d1   nemotron-embedding-ms            Up 4 seconds  (healthy)
6d62008a9b42   compose-nemotron-ocr-1      Up 10 minutes
969b9f5c987c   compose-table-structure-1        Up 10 minutes
```

> **Note:** Container IDs and exact uptimes will vary. The important thing is that all containers are running and the ones marked "(healthy)" show that status.

In [ ]:
check_containers()

---

# Section 3: Test the Services

---

### 3.1 Health Checks

In [ ]:
await check_health(RAG_BASE_URL, "RAG Server")
await check_health(INGESTOR_BASE_URL, "Ingestor Server")

### 3.2 Test LLM (without RAG)

In [ ]:
print("Q: What is 2+2?")
print("A: ", end="")
await chat("What is 2+2?", use_rag=False)

---

# Section 4: Use the Chatbot with Documents

---

Now let's upload documents and ask questions about them.

**In this section, you'll:**
1. Download sample Wikipedia articles (AI and Machine Learning)
2. Create a vector database collection
3. Upload and verify documents are indexed
4. Study the extracted results (text, images, tables)
5. Test the search endpoint directly
6. Compare LLM responses with and without RAG
7. Ask your own questions

### 4.1 Download Sample Document

In [ ]:
# Download 2 sample documents from Wikipedia
SAMPLE_DOCS = [
    ("sample_ai_article.pdf", "Artificial_intelligence"),
    ("sample_ml_article.pdf", "Machine_learning")
]

for filename, topic in SAMPLE_DOCS:
    !curl -sL "https://en.wikipedia.org/api/rest_v1/page/pdf/{topic}" -o {filename}
    print(f"✅ Downloaded: {filename}")

!ls -lh *.pdf

### 4.2 Create Collection

Create the collection before uploading documents:

In [ ]:
# Create the collection first (required before uploading)
COLLECTION_NAME = "multimodal_data"
await create_collection(COLLECTION_NAME)

**Verify Collection Created:**

In [ ]:
# List all collections in the vector database
collections = await list_collections()

# Show raw data
print(f"\nRaw data: {collections}")

### 4.3 Upload Documents


In [ ]:
import asyncio

# Upload all documents in a single request
DOC_FILES = [filename for filename, _ in SAMPLE_DOCS]
print(f"Uploading {len(DOC_FILES)} documents to collection: {COLLECTION_NAME}\n")

response = await upload_documents(DOC_FILES, collection=COLLECTION_NAME)
task_id = response.get("task_id")

if task_id:
    print(f"\nWaiting for upload to complete...")
    for _ in range(24):  # Wait up to 2 minutes
        await asyncio.sleep(5)
        status = await check_upload_status(task_id)
        if status.get("state") == "FINISHED":
            print("\n✅ Upload complete!")
            break
        elif status.get("state") == "FAILED":
            print("\n❌ Upload failed!")
            break
else:
    print("❌ No task_id returned - check ingestor logs")

### 4.4 Verify Upload


In [ ]:
await list_documents(collection=COLLECTION_NAME)

### 4.5 Study Extracted Results

When `APP_NVINGEST_SAVETODISK=True` is enabled, the ingestion pipeline saves the extracted results inside the `rag-vol-ingestor` Docker named volume at:
```
/data/nv-ingest-results/{collection_name}/{filename}.results.jsonl.gz   # path inside the ingestor-server container
```

On the Docker host the same files live under `/var/lib/docker/volumes/rag-vol-ingestor/_data/nv-ingest-results/…`. Because Docker named volumes are owned by root on the host, the next cell copies the results out into a local `ingestor-results/` directory so the notebook can read them as a regular user.

Each `.results.jsonl.gz` file contains the extracted content from the document including:
- **Text chunks**: Extracted text segments with metadata
- **Tables**: Structured table data extracted from the document
- **Images**: Base64-encoded images with optional captions
- **Charts**: Detected charts with extracted data

Let's explore the structure of one result file:

In [ ]:
import gzip
import json
import shutil
import subprocess
from pathlib import Path
from IPython.display import display, HTML, Image
import base64

# Copy the ingestor-server results out of the `rag-vol-ingestor` Docker named volume
# into a local directory so this notebook can read them as the current user.
RESULTS_BASE_DIR = Path("ingestor-results/nv-ingest-results")
RESULTS_BASE_DIR.parent.mkdir(exist_ok=True)
if RESULTS_BASE_DIR.exists():
    shutil.rmtree(RESULTS_BASE_DIR.parent)
    RESULTS_BASE_DIR.parent.mkdir(exist_ok=True)
subprocess.check_call([
    "docker", "run", "--rm",
    "-v", "rag-vol-ingestor:/src:ro",
    "-v", f"{Path.cwd() / RESULTS_BASE_DIR.parent}:/dst",
    "alpine",
    "sh", "-c",
    "cp -a /src/nv-ingest-results/. /dst/nv-ingest-results/ && chown -R $(stat -c '%u:%g' /dst) /dst",
])

COLLECTION_TO_STUDY = COLLECTION_NAME  # Uses the collection name from earlier cells

# List available result files for this collection
collection_results_dir = RESULTS_BASE_DIR / COLLECTION_TO_STUDY
if collection_results_dir.exists():
    result_files = list(collection_results_dir.glob("*.results.jsonl.gz"))
    print(f"📁 Found {len(result_files)} result file(s) in collection '{COLLECTION_TO_STUDY}':\n")
    for f in result_files:
        print(f"  • {f.name}")
else:
    result_files = []
    print(f"❌ Results directory not found: {collection_results_dir}")
    print("   Make sure APP_NVINGEST_SAVETODISK=True was set before ingestion.")

**Generate PDF with Extracted Content:**

This cell creates a PDF containing all extracted text and images from the document.

In [ ]:
import gzip
import json
from pathlib import Path
import base64
import os
import sys

# Install fpdf2 in the correct environment
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "fpdf2", "-q"])

from fpdf import FPDF

# Create results_study folder
RESULTS_STUDY_DIR = Path("results_study")
RESULTS_STUDY_DIR.mkdir(exist_ok=True)

# Path to the specific result file (adjust path as needed)
RESULT_FILE = Path("ingestor-results/nv-ingest-results") / COLLECTION_NAME / "sample_ai_article.pdf.results.jsonl.gz"

# Load the file
extracted_content = []
try:
    with gzip.open(RESULT_FILE, 'rt', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                extracted_content.append(json.loads(line))
except gzip.BadGzipFile:
    with open(str(RESULT_FILE).replace('.gz', ''), 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                extracted_content.append(json.loads(line))

print(f"Loaded {len(extracted_content)} items from {RESULT_FILE.name}")

# Save full JSON results
output_json = RESULTS_STUDY_DIR / "sample_ai_article_extracted.json"
with open(output_json, 'w', encoding='utf-8') as f:
    json.dump(extracted_content, f, indent=2)

# Create PDF with extracted content
class PDF(FPDF):
    def header(self):
        self.set_font('Helvetica', 'B', 12)
        self.cell(0, 10, 'Extracted Content: sample_ai_article.pdf', border=False, ln=True, align='C')
        self.ln(5)
    
    def footer(self):
        self.set_y(-15)
        self.set_font('Helvetica', 'I', 8)
        self.cell(0, 10, f'Page {self.page_no()}', align='C')

pdf = PDF()
pdf.set_auto_page_break(auto=True, margin=15)
pdf.add_page()

# Content breakdown
content_types = {}
for item in extracted_content:
    doc_type = item.get('document_type', 'unknown')
    content_types[doc_type] = content_types.get(doc_type, 0) + 1

pdf.set_font('Helvetica', 'B', 11)
pdf.cell(0, 8, f'Total extracted items: {len(extracted_content)}', ln=True)
pdf.ln(3)
pdf.set_font('Helvetica', '', 10)
for doc_type, count in sorted(content_types.items()):
    pdf.cell(0, 6, f'  - {doc_type}: {count}', ln=True)
pdf.ln(5)
pdf.cell(0, 0, '', border='T', ln=True)
pdf.ln(10)

# Add each item
for i, item in enumerate(extracted_content):
    doc_type = item.get('document_type', 'unknown')
    
    # Section header
    pdf.set_font('Helvetica', 'B', 10)
    pdf.set_fill_color(230, 230, 230)
    pdf.cell(0, 8, f'[Item {i+1}] Type: {doc_type}', ln=True, fill=True)
    pdf.ln(3)
    
    if 'metadata' in item:
        meta = item['metadata']
        
        # Handle text content
        if 'content' in meta and doc_type == 'text':
            pdf.set_font('Helvetica', '', 9)
            content = meta['content']
            content = content.encode('latin-1', 'replace').decode('latin-1')
            pdf.multi_cell(0, 5, content)
            pdf.ln(5)
        
        # Handle images
        elif doc_type == 'image' and 'content' in meta:
            try:
                img_data = base64.b64decode(meta['content'])
                temp_img = RESULTS_STUDY_DIR / f"temp_img_{i}.png"
                with open(temp_img, 'wb') as img_f:
                    img_f.write(img_data)
                
                pdf.image(str(temp_img), w=min(180, pdf.epw))
                
                if 'image_metadata' in meta and meta['image_metadata'].get('caption'):
                    pdf.set_font('Helvetica', 'I', 8)
                    caption = meta['image_metadata']['caption']
                    caption = caption.encode('latin-1', 'replace').decode('latin-1')
                    pdf.multi_cell(0, 4, f"Caption: {caption}")
                
                temp_img.unlink()
                pdf.ln(5)
            except Exception as e:
                pdf.set_font('Helvetica', 'I', 9)
                pdf.cell(0, 6, f'[Image could not be rendered: {e}]', ln=True)
        
        # Handle tables/structured content
        elif doc_type == 'structured' and 'content' in meta:
            pdf.set_font('Courier', '', 8)
            content = meta['content'][:2000]
            content = content.encode('latin-1', 'replace').decode('latin-1')
            pdf.multi_cell(0, 4, content)
            pdf.ln(5)
    
    pdf.ln(3)

# Save PDF
output_pdf = RESULTS_STUDY_DIR / "sample_ai_article_extracted.pdf"
pdf.output(str(output_pdf))

print(f"\nResults saved to '{RESULTS_STUDY_DIR}/' folder:")
print(f"   - {output_json.name} (full JSON)")
print(f"   - {output_pdf.name} (PDF with text & images)")

After running the cell above, open the `results_study/` folder to explore:

- **sample_ai_article_extracted.json** - Raw JSON with all extracted data (text, images, metadata)
- **sample_ai_article_extracted.pdf** - Formatted PDF showing text content and embedded images

You can modify the `RESULT_FILE` path to study other documents in your collection.

### 4.6 Test Search Endpoint (OpenAI-Compatible)


In [ ]:
async def search(query: str, collection: str = "multimodal_data", top_k: int = 3) -> dict:
    """Search for relevant document chunks without generating an answer."""
    payload = {
        "query": query,
        "collection_names": [collection],
        "reranker_top_k": top_k
    }
    
    async with aiohttp.ClientSession() as session:
        async with session.post(f"{RAG_BASE_URL}/v1/search", json=payload) as resp:
            return await resp.json()

import base64
from IPython.display import display, Image as IPImage

def is_base64_image(content: str) -> bool:
    """Check if content is base64 image data."""
    return content.startswith(("/9j/", "iVBOR", "data:image"))

def display_content(content: str, max_len: int = 200):
    """Display content - renders images, truncates text."""
    if is_base64_image(content):
        # Decode and display image
        try:
            # Handle data:image prefix if present
            if content.startswith("data:image"):
                content = content.split(",", 1)[1]
            img_data = base64.b64decode(content)
            display(IPImage(data=img_data, width=400))
        except Exception as e:
            print(f"    [Image decode error: {e}]")
    else:
        # Display text
        text = content[:max_len].replace("\n", " ").strip()
        text = text + "..." if len(content) > max_len else text
        print(f"    {text}")

# Test the search endpoint
print("Testing /v1/search endpoint...")
print("=" * 60)
results = await search("What is artificial intelligence?", top_k=3)

print(f"Found {len(results.get('results', []))} results:\n")
for i, result in enumerate(results.get("results", []), 1):
    score = result.get("score", 0)
    content = result.get("content", "")
    doc_name = result.get("document_name", "unknown")
    content_type = result.get("type", "text")
    
    print(f"[{i}] Score: {score:.4f} | Doc: {doc_name} | Type: {content_type}")
    display_content(content)
    print()

### 4.7 Compare: LLM vs RAG


In [ ]:
QUESTION = "What are the main approaches to artificial intelligence?"

print("=" * 60)
print("WITHOUT RAG (LLM general knowledge only)")
print("=" * 60)
print(f"Q: {QUESTION}")
print("A: ", end="")
await chat(QUESTION, use_rag=False)

print()
print("=" * 60)
print("WITH RAG (searches your documents first)")
print("=" * 60)
print(f"Q: {QUESTION}")
print("A: ", end="")
await chat(QUESTION, use_rag=True)

### 4.8 Ask Your Own Questions


In [ ]:
# Edit this question and run the cell
my_question = "What is machine learning?"

print(f"Q: {my_question}")
print("A: ", end="")
await chat(my_question, use_rag=True)

---

# Section 5: Cleanup

---

### 5.1 Delete Uploaded Documents

In [ ]:
# Delete all uploaded documents
doc_names = [filename for filename, _ in SAMPLE_DOCS]
await delete_document(doc_names, collection=COLLECTION_NAME)

**Delete Collections:**

In [ ]:
# List all collections and delete each one
collections = await list_collections()

if collections:
    print(f"\nDeleting {len(collections)} collection(s)...")
    for c in collections:
        name = c.get("collection_name", c) if isinstance(c, dict) else c
        await delete_collection(name)
    
    print("\n✅ All collections deleted")
    
    # Verify
    await list_collections()
else:
    print("No collections to delete")

### 5.2 Stop All Services

In [ ]:
stop_all()

---

# Quick Reference

---

## Function Cheat Sheet

| Function | Description |
|----------|-------------|
| `deploy_all()` | Start all services |
| `stop_all()` | Stop all services |
| `check_containers()` | List running containers |
| `await chat(question)` | Ask a question (`use_rag=True` by default) |
| `await chat(question, use_rag=False)` | Ask LLM directly without document search |
| `await upload_documents(paths)` | Upload file(s) or directory to knowledge base |
| `await list_documents()` | List uploaded files |
| `await delete_document(name)` | Remove a file |
| `await list_collections()` | List all collections |

---

## Troubleshooting

| Issue | Solution |
|-------|----------|
| Containers not starting | Run `docker logs <container-name>` to see errors |
| Out of GPU memory | Run `stop_all()`, or use NVIDIA-hosted LLM |
| API not responding | Wait for "healthy" status, check `docker logs rag-server` |
| Upload stuck | Check `docker logs ingestor-server` |

---

## Service Ports

| Service | Port |
|---------|------|
| RAG Server | 8081 |
| Ingestor Server | 8082 |
| RAG Frontend | 8090 |
| Elasticsearch | 9200 |

---

# Next Steps

---

Now that you've deployed the RAG Blueprint and tested basic functionality, explore these notebooks to learn more advanced features:

## 🟢 Beginner - Learn the APIs

| Notebook | Description |
|----------|-------------|
| **ingestion_api_usage.ipynb** | Deep dive into the ingestion service - upload, process, and manage documents |
| **retriever_api_usage.ipynb** | Explore query techniques, retrieval strategies, and search parameters |

## 🟡 Intermediate - Extend Functionality

| Notebook | Description |
|----------|-------------|
| **summarization.ipynb** | Document summarization with page filtering and multiple strategies |
| **nb_metadata.ipynb** | Metadata ingestion, filtering, and extraction for enhanced retrieval |
| **rag_library_usage.ipynb** | Native Python client usage - full end-to-end API examples |
| **rag_library_lite_usage.ipynb** | Containerless deployment with Milvus Lite (no Docker required) |
| **evaluation_01_ragas.ipynb** | Evaluate RAG quality using RAGAS metrics |
| **evaluation_02_recall.ipynb** | Measure retrieval recall at various top-k thresholds |

## 🔴 Advanced - Build & Customize

| Notebook | Description |
|----------|-------------|
| **image_input.ipynb** | Multimodal queries with text + images (VLM embeddings) |
| **building_rag_vdb_operator.ipynb** | Create custom vector database operators (e.g., OpenSearch) |
| **mcp_server_usage.ipynb** | Use Model Context Protocol (MCP) transports instead of REST APIs |

## 📚 Additional Resources

- **Documentation**: https://docs.nvidia.com/rag/latest/
- **GitHub**: https://github.com/NVIDIA-AI-Blueprints/rag
- **Support Matrix**: https://docs.nvidia.com/rag/latest/support-matrix.html

---

**Happy building! 🚀**

---

# Section 6: Darla Command Center Demo

---

This section adds a lightweight **agentic command center** flow you can use in a hackathon demo.

It turns messy customer asks into:
- objective + context extraction
- readiness score by category
- blocker list with severity
- executive brief + engineering checklist + POC report

In [ ]:
from dataclasses import dataclass
from datetime import datetime
from typing import Dict, List
import textwrap
import re


@dataclass
class DarlaResult:
    objective: str
    readiness: Dict[str, int]
    blockers: List[dict]
    next_actions: List[str]
    confidence: int


def _contains_any(text: str, phrases: List[str]) -> bool:
    lowered = text.lower()
    return any(p.lower() in lowered for p in phrases)


def intake_agent(customer_input: str) -> dict:
    cleaned = re.sub(r"\s+", " ", customer_input.strip())
    objective = cleaned if len(cleaned) <= 220 else cleaned[:217] + "..."
    return {
        "raw": cleaned,
        "objective": objective,
    }


def decomposition_agent(intake: dict) -> dict:
    text = intake["raw"].lower()
    workstreams = [
        "Business outcome",
        "Data readiness",
        "Security and compliance",
        "Infrastructure and deployment",
        "Stakeholder ownership",
        "Timeline and delivery plan",
    ]

    if "rag" in text or "retrieval" in text:
        workstreams.append("Retrieval quality and citation fidelity")
    if _contains_any(text, ["image", "multimodal", "vlm"]):
        workstreams.append("Multimodal ingestion and inference")
    if _contains_any(text, ["agent", "workflow", "orchestration"]):
        workstreams.append("Agent orchestration and guardrails")

    return {"workstreams": workstreams}


def readiness_agent(intake: dict) -> Dict[str, int]:
    text = intake["raw"]
    has_data = _contains_any(text, ["data", "documents", "pdf", "knowledge base", "catalog"])
    has_security = _contains_any(text, ["security", "compliance", "pii", "governance", "policy"])
    has_owner = _contains_any(text, ["owner", "team", "stakeholder", "sponsor"])
    has_timeline = _contains_any(text, ["week", "sprint", "timeline", "deadline", "q3", "q4"])
    has_budget = _contains_any(text, ["budget", "cost", "roi", "funding"])
    has_infra = _contains_any(text, ["gpu", "cluster", "docker", "kubernetes", "helm", "deploy"])

    scores = {
        "Business": 80 if has_budget else 58,
        "Data": 82 if has_data else 52,
        "Security": 80 if has_security else 48,
        "Infrastructure": 83 if has_infra else 55,
        "Ownership": 78 if has_owner else 50,
        "Timeline": 76 if has_timeline else 52,
    }
    return scores


def risk_agent(scores: Dict[str, int]) -> List[dict]:
    blockers = []
    for category, score in scores.items():
        if score < 60:
            severity = "high"
        elif score < 75:
            severity = "medium"
        else:
            continue

        blockers.append(
            {
                "category": category,
                "severity": severity,
                "message": f"{category} readiness is {score}/100 and needs mitigation before production rollout.",
            }
        )
    return blockers


def evidence_agent(intake: dict, workstreams: List[str], blockers: List[dict]) -> List[str]:
    evidence = [
        f"Input objective extracted from customer intake: '{intake['objective']}'",
        f"Derived workstreams: {', '.join(workstreams[:4])}{'...' if len(workstreams) > 4 else ''}",
    ]
    if blockers:
        evidence.append(
            "Blockers identified from low-readiness categories: "
            + ", ".join(f"{b['category']} ({b['severity']})" for b in blockers)
        )
    else:
        evidence.append("No major blockers identified from readiness scoring.")
    return evidence


def output_agent(intake: dict, scores: Dict[str, int], blockers: List[dict], evidence: List[str]) -> DarlaResult:
    avg_score = int(sum(scores.values()) / len(scores))
    confidence = max(55, min(95, avg_score))

    actions = [
        "Run stakeholder alignment workshop and confirm business KPI target.",
        "Validate data availability and quality for top 3 user journeys.",
        "Define security controls and approval path for pilot deployment.",
        "Stand up a 2-week POC plan with named owner per workstream.",
    ]

    if any(b["category"] == "Security" for b in blockers):
        actions.insert(0, "Schedule security and compliance review before ingestion at scale.")

    return DarlaResult(
        objective=intake["objective"],
        readiness=scores,
        blockers=blockers,
        next_actions=actions,
        confidence=confidence,
    )


def run_darla_command_center(customer_input: str) -> DarlaResult:
    intake = intake_agent(customer_input)
    decomposed = decomposition_agent(intake)
    scores = readiness_agent(intake)
    blockers = risk_agent(scores)
    evidence = evidence_agent(intake, decomposed["workstreams"], blockers)
    result = output_agent(intake, scores, blockers, evidence)

    print("=" * 72)
    print("DARLA COMMAND CENTER")
    print("=" * 72)
    print(f"Timestamp: {datetime.utcnow().isoformat(timespec='seconds')}Z")
    print(f"Objective: {result.objective}")
    print(f"Confidence: {result.confidence}/100")
    print("\nReadiness Scorecard:")
    for k, v in result.readiness.items():
        status = "GREEN" if v >= 75 else "YELLOW" if v >= 60 else "RED"
        print(f"  - {k:<15} {v:>3}/100  [{status}]")

    print("\nBlockers:")
    if result.blockers:
        for b in result.blockers:
            print(f"  - ({b['severity'].upper()}) {b['category']}: {b['message']}")
    else:
        print("  - None")

    print("\nSuggested Next Actions:")
    for idx, action in enumerate(result.next_actions, start=1):
        print(f"  {idx}. {action}")

    print("=" * 72)
    return result


def render_executive_brief(result: DarlaResult) -> str:
    avg = int(sum(result.readiness.values()) / len(result.readiness))
    top_risks = ", ".join(f"{b['category']} ({b['severity']})" for b in result.blockers) or "No critical blockers"
    brief = f"""
    DARLA EXECUTIVE BRIEF

    Objective:
    {result.objective}

    Overall readiness: {avg}/100
    Confidence: {result.confidence}/100

    Top risks:
    {top_risks}

    Recommended next actions:
    - {result.next_actions[0]}
    - {result.next_actions[1]}
    - {result.next_actions[2]}
    """
    return textwrap.dedent(brief).strip()


def render_engineering_checklist(result: DarlaResult) -> List[str]:
    checklist = [
        "[ ] Confirm deployment target (Docker Compose, Helm, or managed endpoint).",
        "[ ] Validate vector DB + collection strategy for pilot corpus.",
        "[ ] Configure embedding, reranking, and generation model endpoints.",
        "[ ] Add ingestion quality checks and citation validation.",
        "[ ] Define guardrails, policy checks, and observability baseline.",
    ]
    if any(b["category"] == "Infrastructure" for b in result.blockers):
        checklist.insert(0, "[ ] Resolve GPU/compute capacity plan before load testing.")
    return checklist


print("✅ Darla command center helpers loaded")

In [ ]:
from typing import Any


async def collect_rag_grounding(customer_input: str, collection: str | None = None, top_k: int = 3) -> dict[str, Any]:
    active_collection = collection or globals().get("COLLECTION_NAME", "multimodal_data")
    queries = [
        customer_input,
        f"site survey BOM deployment risk {customer_input}",
        "RAG notebook launchable deployment summary",
    ]

    snippets: list[str] = []
    cited_artifacts: list[str] = []

    for query in queries:
        try:
            result = await search(query, collection=active_collection, top_k=top_k)
        except Exception as exc:
            snippets.append(f"- RAG search failed for '{query}': {exc}")
            continue

        matches = result.get("results", [])
        if not isinstance(matches, list) or not matches:
            snippets.append(f"- No direct hits for: {query}")
            continue

        for item in matches[:top_k]:
            doc_name = str(item.get("document_name", "unknown"))
            score = float(item.get("score", 0) or 0)
            content = str(item.get("content", "")).replace("\n", " ").strip()
            preview = content[:220]
            snippets.append(f"- {doc_name} [{score:.3f}]: {preview}")
            cited_artifacts.append(doc_name)

    try:
        summary = await chat(
            f"Summarize relevant operational context for: {customer_input}",
            use_rag=True,
            collection=active_collection,
            show_citations=True,
        )
    except Exception as exc:
        summary = f"RAG chat failed: {exc}"

    if not snippets:
        snippets.append("- No grounding snippets found.")

    return {
        "collection": active_collection,
        "snippets": snippets,
        "citations": sorted(set(cited_artifacts)),
        "summary": summary,
    }


async def run_darla_command_center(customer_input: str) -> DarlaResult:
    intake = intake_agent(customer_input)
    decomposed = decomposition_agent(intake)
    scores = readiness_agent(intake)
    blockers = risk_agent(scores)
    rag_context = await collect_rag_grounding(customer_input, collection=globals().get("COLLECTION_NAME"))
    evidence = evidence_agent(intake, decomposed["workstreams"], blockers)
    evidence.extend([f"RAG collection: {rag_context['collection']}"])
    evidence.extend([f"RAG grounding: {item}" for item in rag_context["snippets"]])
    if rag_context["citations"]:
        evidence.append("Cited artifacts: " + ", ".join(rag_context["citations"]))

    result = output_agent(intake, scores, blockers, evidence)

    print("\n" + "=" * 72)
    print("RAG GROUNDING")
    print("=" * 72)
    print(rag_context["summary"])
    print("\nRetrieved snippets:")
    for item in rag_context["snippets"]:
        print(f"  {item}")

    print("\n" + "=" * 72)
    print("DARLA COMMAND CENTER")
    print("=" * 72)
    print(f"Timestamp: {datetime.utcnow().isoformat(timespec='seconds')}Z")
    print(f"Objective: {result.objective}")
    print(f"Confidence: {result.confidence}/100")
    print("\nReadiness Scorecard:")
    for key, value in result.readiness.items():
        status = "GREEN" if value >= 75 else "YELLOW" if value >= 60 else "RED"
        print(f"  - {key:<15} {value:>3}/100  [{status}]")

    print("\nBlockers:")
    if result.blockers:
        for blocker in result.blockers:
            print(f"  - ({blocker['severity'].upper()}) {blocker['category']}: {blocker['message']}")
    else:
        print("  - None")

    print("\nSuggested Next Actions:")
    for idx, action in enumerate(result.next_actions, start=1):
        print(f"  {idx}. {action}")

    if evidence:
        print("\nEvidence Trail:")
        for item in evidence[:10]:
            print(f"  - {item}")

    print("=" * 72)
    return result


### 6.1 Run Darla on a Customer Ask

Edit the sample input, run the cell, and Darla will generate readiness + blockers + actions.

In [ ]:
customer_ask = """
We want a secure enterprise RAG assistant for support engineers.
Need on-prem or hybrid deployment with NVIDIA stack, rollout in one quarter,
and clear ownership across sales engineering and delivery teams.
We also need citations and a checklist for POC readiness.
""".strip()

darla_result = await run_darla_command_center(customer_ask)


### 6.2 Export Demo Outputs

These outputs are formatted for your 5-minute presentation flow.

In [ ]:
import json

executive_brief = render_executive_brief(darla_result)
engineering_checklist = render_engineering_checklist(darla_result)

print("\n" + "=" * 72)
print(executive_brief)
print("=" * 72)

print("\nENGINEERING CHECKLIST")
for item in engineering_checklist:
    print(item)

poc_readiness_report = {
    "project": "Darla Command Center",
    "objective": darla_result.objective,
    "readiness": darla_result.readiness,
    "confidence": darla_result.confidence,
    "blockers": darla_result.blockers,
    "next_actions": darla_result.next_actions,
}

print("\nPOC READINESS REPORT (JSON)")
print(json.dumps(poc_readiness_report, indent=2))

---

# Section 7: AIDOS Quick Run (LaunchPad + API Keys)

---

This section is the direct AIDOS path (no Darla dependency).

It does three things:
1. Configure LaunchPad/API key environment
2. Run AIDOS intake -> SoT -> validation -> NetBox payload -> deployment plan -> ansible bundle
3. Optionally run the full workflow writer for output artifacts


In [ ]:
import os
import getpass
from pathlib import Path


def _ensure_env_key(name: str, prefix: str | None = None) -> None:
    current = os.environ.get(name, "")
    if current:
        print(f"✅ {name} already set")
        return

    value = getpass.getpass(f"Enter {name}: ").strip()
    if prefix and value and not value.startswith(prefix):
        raise ValueError(f"{name} must start with '{prefix}'")
    os.environ[name] = value
    print(f"✅ {name} set")


# LaunchPad / NVIDIA credentials
_ensure_env_key("NGC_API_KEY", prefix="nvapi-")

# Optional but recommended for hosted inference/evaluation endpoints
if not os.environ.get("NVIDIA_API_KEY"):
    print("ℹ️ NVIDIA_API_KEY not set. If your environment uses hosted API calls, set it now.")
    maybe_key = getpass.getpass("Enter NVIDIA_API_KEY (optional, press Enter to skip): ").strip()
    if maybe_key:
        os.environ["NVIDIA_API_KEY"] = maybe_key
        print("✅ NVIDIA_API_KEY set")

print("\nEnvironment readiness:")
print(f"- NGC_API_KEY: {'set' if bool(os.environ.get('NGC_API_KEY')) else 'missing'}")
print(f"- NVIDIA_API_KEY: {'set' if bool(os.environ.get('NVIDIA_API_KEY')) else 'missing'}")


In [ ]:
import json
import os
from pathlib import Path

from aidos.ingestion import ingest_inputs
from aidos.truth import build_canonical_sot
from aidos.validation import validate_sot
from aidos.netbox_sync import build_netbox_payload
from aidos.planning import build_runbook_plan, build_ansible_bundle, build_ansible_playbook, build_agentic_task_graph
from aidos.orchestrator import run_mvp_workflow


def _normalize_path(value: str | None) -> str | None:
    if value is None:
        return None
    text = str(value).strip()
    return text or None


def _resolve_manual_or_candidates(
    manual_path: str | None,
    roots: list[Path],
    candidate_names: tuple[str, ...],
) -> str | None:
    manual = _normalize_path(manual_path)
    if manual:
        p = Path(manual)
        if p.exists():
            return str(p)
        print(f"WARNING: manual path does not exist: {manual}")
        return None

    for root in roots:
        for name in candidate_names:
            candidate = (root / name).resolve()
            if candidate.exists():
                return str(candidate)
    return None


def refresh_aidos_inputs(
    input_root: str | None = None,
    output_dir: str | None = None,
    survey_path: str | None = None,
    bom_path: str | None = None,
    workload_path: str | None = None,
    context_path: str | None = None,
    network_layout_path: str | None = None,
    observed_path: str | None = None,
    netbox_base_url: str | None = None,
    netbox_token: str | None = None,
    verbose: bool = True,
) -> None:
    global AIDOS_INPUT_ROOT, AIDOS_OUTPUT_DIR
    global AIDOS_SURVEY_PATH, AIDOS_BOM_PATH, AIDOS_WORKLOAD_PATH
    global AIDOS_CONTEXT_PATH, AIDOS_NETWORK_LAYOUT_PATH, AIDOS_OBSERVED_PATH
    global AIDOS_NETBOX_BASE_URL, AIDOS_NETBOX_TOKEN, AIDOS_READY, AIDOS_MISSING_INPUTS

    chosen_root = _normalize_path(input_root) or os.environ.get("AIDOS_INPUT_ROOT", ".")
    AIDOS_INPUT_ROOT = Path(chosen_root)

    AIDOS_OUTPUT_DIR = _normalize_path(output_dir) or os.environ.get(
        "AIDOS_OUTPUT_DIR", "aidos/outputs/versatile_run"
    )

    roots_to_search = [AIDOS_INPUT_ROOT, Path("."), Path("fakedata"), Path("aidos/examples")]

    AIDOS_SURVEY_PATH = _resolve_manual_or_candidates(
        survey_path or os.environ.get("AIDOS_SURVEY_PATH"),
        roots_to_search,
        ("survey.json", "site_survey.json", "site_survey.yaml", "site_survey.yml"),
    )
    AIDOS_BOM_PATH = _resolve_manual_or_candidates(
        bom_path or os.environ.get("AIDOS_BOM_PATH"),
        roots_to_search,
        ("bom.yaml", "bom.yml", "bom.json"),
    )
    AIDOS_WORKLOAD_PATH = _resolve_manual_or_candidates(
        workload_path or os.environ.get("AIDOS_WORKLOAD_PATH"),
        roots_to_search,
        ("workload.yaml", "workload.yml", "workload.json"),
    )
    AIDOS_CONTEXT_PATH = _resolve_manual_or_candidates(
        context_path or os.environ.get("AIDOS_CONTEXT_PATH"),
        roots_to_search,
        ("context.json", "context.yaml", "context.yml"),
    )
    AIDOS_NETWORK_LAYOUT_PATH = _resolve_manual_or_candidates(
        network_layout_path or os.environ.get("AIDOS_NETWORK_LAYOUT_PATH"),
        roots_to_search,
        ("network_layout.xlsx", "network_layout.csv", "FAKEDATA.xlsx"),
    )
    AIDOS_OBSERVED_PATH = _resolve_manual_or_candidates(
        observed_path or os.environ.get("AIDOS_OBSERVED_PATH"),
        roots_to_search,
        ("observed.json",),
    )

    AIDOS_NETBOX_BASE_URL = _normalize_path(netbox_base_url) or os.environ.get(
        "AIDOS_NETBOX_BASE_URL", "https://xvkj6149.cloud.netboxapp.com"
    )
    AIDOS_NETBOX_TOKEN = _normalize_path(netbox_token)
    if AIDOS_NETBOX_TOKEN is None:
        AIDOS_NETBOX_TOKEN = _normalize_path(os.environ.get("NB_TOKEN", "")) or ""

    AIDOS_MISSING_INPUTS = []
    if AIDOS_SURVEY_PATH is None:
        AIDOS_MISSING_INPUTS.append("survey.json (or site_survey.json/yaml)")
    if AIDOS_BOM_PATH is None and AIDOS_WORKLOAD_PATH is None:
        AIDOS_MISSING_INPUTS.append("bom.(yaml/yml/json) or workload.(yaml/yml/json)")

    AIDOS_READY = len(AIDOS_MISSING_INPUTS) == 0

    if verbose:
        print("AIDOS inputs:")
        print(f"- input_root: {AIDOS_INPUT_ROOT}")
        print(f"- survey: {AIDOS_SURVEY_PATH}")
        print(f"- bom: {AIDOS_BOM_PATH}")
        print(f"- workload: {AIDOS_WORKLOAD_PATH}")
        print(f"- context: {AIDOS_CONTEXT_PATH}")
        print(f"- network_layout: {AIDOS_NETWORK_LAYOUT_PATH}")
        print(f"- observed: {AIDOS_OBSERVED_PATH}")
        print(f"- output_dir: {AIDOS_OUTPUT_DIR}")

        if AIDOS_READY:
            print("\nPreflight: READY")
        else:
            print("\nPreflight: BLOCKED")
            print("Missing required inputs:")
            for item in AIDOS_MISSING_INPUTS:
                print(f"- {item}")

        print("\nNetBox config:")
        print(f"- base_url: {AIDOS_NETBOX_BASE_URL}")
        print(f"- token: {'set' if bool(AIDOS_NETBOX_TOKEN) else 'not set (uses backend env if available)'}")


def clear_path_settings() -> None:
    """Clear general path settings and re-resolve from defaults/env."""
    refresh_aidos_inputs(input_root='.', output_dir='aidos/outputs/versatile_run', netbox_base_url='', verbose=True)


def clear_file_selections() -> None:
    """Clear manual file selections and re-resolve from search roots."""
    refresh_aidos_inputs(
        survey_path='',
        bom_path='',
        workload_path='',
        context_path='',
        network_layout_path='',
        observed_path='',
        verbose=True,
    )


# Initialize once so downstream cells can run without widget interaction.
refresh_aidos_inputs(verbose=True)


try:
    import ipywidgets as widgets
    from IPython.display import display

    root_text = widgets.Text(value=str(AIDOS_INPUT_ROOT), description="Input Root", layout=widgets.Layout(width="650px"))
    out_text = widgets.Text(value=AIDOS_OUTPUT_DIR, description="Output Dir", layout=widgets.Layout(width="650px"))

    survey_text = widgets.Text(value=AIDOS_SURVEY_PATH or "", description="Survey", layout=widgets.Layout(width="850px"))
    bom_text = widgets.Text(value=AIDOS_BOM_PATH or "", description="BOM", layout=widgets.Layout(width="850px"))
    workload_text = widgets.Text(value=AIDOS_WORKLOAD_PATH or "", description="Workload", layout=widgets.Layout(width="850px"))
    context_text = widgets.Text(value=AIDOS_CONTEXT_PATH or "", description="Context", layout=widgets.Layout(width="850px"))
    network_text = widgets.Text(value=AIDOS_NETWORK_LAYOUT_PATH or "", description="Network", layout=widgets.Layout(width="850px"))
    observed_text = widgets.Text(value=AIDOS_OBSERVED_PATH or "", description="Observed", layout=widgets.Layout(width="850px"))

    nb_base_text = widgets.Text(value=AIDOS_NETBOX_BASE_URL, description="NetBox URL", layout=widgets.Layout(width="850px"))
    nb_token_text = widgets.Password(value=AIDOS_NETBOX_TOKEN or "", description="NetBox Token", layout=widgets.Layout(width="850px"))

    apply_btn = widgets.Button(description="Apply", button_style="success")
    clear_paths_btn = widgets.Button(description="Clear Paths", button_style="warning")
    clear_files_btn = widgets.Button(description="Clear Files", button_style="warning")

    out = widgets.Output()

    def _apply_from_widgets(_):
        with out:
            out.clear_output(wait=True)
            refresh_aidos_inputs(
                input_root=root_text.value,
                output_dir=out_text.value,
                survey_path=survey_text.value,
                bom_path=bom_text.value,
                workload_path=workload_text.value,
                context_path=context_text.value,
                network_layout_path=network_text.value,
                observed_path=observed_text.value,
                netbox_base_url=nb_base_text.value,
                netbox_token=nb_token_text.value,
                verbose=True,
            )

    def _clear_paths(_):
        root_text.value = "."
        out_text.value = "aidos/outputs/versatile_run"
        nb_base_text.value = os.environ.get("AIDOS_NETBOX_BASE_URL", "https://xvkj6149.cloud.netboxapp.com")
        with out:
            out.clear_output(wait=True)
            refresh_aidos_inputs(
                input_root=root_text.value,
                output_dir=out_text.value,
                survey_path=survey_text.value,
                bom_path=bom_text.value,
                workload_path=workload_text.value,
                context_path=context_text.value,
                network_layout_path=network_text.value,
                observed_path=observed_text.value,
                netbox_base_url=nb_base_text.value,
                netbox_token=nb_token_text.value,
                verbose=True,
            )

    def _clear_files(_):
        survey_text.value = ""
        bom_text.value = ""
        workload_text.value = ""
        context_text.value = ""
        network_text.value = ""
        observed_text.value = ""
        with out:
            out.clear_output(wait=True)
            refresh_aidos_inputs(
                input_root=root_text.value,
                output_dir=out_text.value,
                survey_path="",
                bom_path="",
                workload_path="",
                context_path="",
                network_layout_path="",
                observed_path="",
                netbox_base_url=nb_base_text.value,
                netbox_token=nb_token_text.value,
                verbose=True,
            )

    apply_btn.on_click(_apply_from_widgets)
    clear_paths_btn.on_click(_clear_paths)
    clear_files_btn.on_click(_clear_files)

    display(widgets.HTML("<b>AIDOS Input Configurator</b>"))
    display(root_text, out_text)
    display(survey_text, bom_text, workload_text, context_text, network_text, observed_text)
    display(nb_base_text, nb_token_text)
    display(widgets.HBox([apply_btn, clear_paths_btn, clear_files_btn]))
    display(out)
except Exception as exc:
    print(f"Widget UI unavailable: {exc}")
    print("Use refresh_aidos_inputs(...) directly to set any paths.")

### 7.1 Run AIDOS Backend for UI

This starts the AIDOS FastAPI backend so you can operate from the browser UI.

- Backend docs: `http://127.0.0.1:8091/docs`
- AIDOS UI: `http://127.0.0.1:8091/ui`
- Health: `http://127.0.0.1:8091/health`

In [ ]:
import subprocess
import socket
import time
import requests

AIDOS_API_HOST = "127.0.0.1"
AIDOS_API_PORT = 8091
AIDOS_API_URL = f"http://{AIDOS_API_HOST}:{AIDOS_API_PORT}"


def _port_is_open(host: str, port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(0.3)
        return sock.connect_ex((host, port)) == 0


def _wait_for_health(base_url: str, timeout_s: int = 30) -> bool:
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            resp = requests.get(f"{base_url}/health", timeout=2)
            if resp.status_code == 200:
                return True
        except Exception:
            pass
        time.sleep(0.5)
    return False


if "AIDOS_API_PROC" in globals() and AIDOS_API_PROC.poll() is None:
    print(f"✅ AIDOS backend already running at {AIDOS_API_URL}")
else:
    if _port_is_open(AIDOS_API_HOST, AIDOS_API_PORT):
        print(f"ℹ️ Port {AIDOS_API_PORT} is already in use. Assuming AIDOS backend is already running.")
    else:
        AIDOS_API_PROC = subprocess.Popen(
            [
                "uv",
                "run",
                "python",
                "-m",
                "uvicorn",
                "aidos.api:app",
                "--host",
                AIDOS_API_HOST,
                "--port",
                str(AIDOS_API_PORT),
            ],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
        )
        print(f"🚀 Starting AIDOS backend on {AIDOS_API_URL}...")

    if _wait_for_health(AIDOS_API_URL):
        print(f"✅ AIDOS backend healthy: {AIDOS_API_URL}/health")
        print(f"🌐 UI: {AIDOS_API_URL}/ui")
        print(f"📘 API docs: {AIDOS_API_URL}/docs")
    else:
        print("❌ AIDOS backend did not become healthy within timeout.")
        if "AIDOS_API_PROC" in globals() and AIDOS_API_PROC.poll() is not None:
            out, _ = AIDOS_API_PROC.communicate()
            print(out[-2000:] if out else "No process output captured.")

In [ ]:
# Optional: run one AIDOS flow through HTTP API (same backend used by /ui).

if not AIDOS_READY:
    print("Skipping flow call: preflight is BLOCKED. Fix missing files in AIDOS_INPUT_ROOT first.")
else:
    flow_payload = {
        "survey_path": AIDOS_SURVEY_PATH,
        "output_dir": AIDOS_OUTPUT_DIR,
        "bom_path": AIDOS_BOM_PATH,
        "workload_path": AIDOS_WORKLOAD_PATH,
        "context_path": AIDOS_CONTEXT_PATH,
        "network_layout_path": AIDOS_NETWORK_LAYOUT_PATH,
        "observed_path": AIDOS_OBSERVED_PATH,
        "sync_netbox": True,
        "netbox_base_url": AIDOS_NETBOX_BASE_URL,
        "netbox_token": AIDOS_NETBOX_TOKEN or None,
        "netbox_dry_run": False,
        "execute": True,
        "auto_approve": False,
    }

    resp = requests.post(f"{AIDOS_API_URL}/v1/flow", json=flow_payload, timeout=240)
    print(f"Status: {resp.status_code}")
    if resp.headers.get("content-type", "").startswith("application/json"):
        flow_result = resp.json()
        print(json.dumps(flow_result, indent=2)[:4000])
    else:
        print(resp.text[:1000])

In [ ]:
# AIDOS multi-agent pass (deterministic control-plane stages)
# sitesurvey agent + bom/workload agent + context agent + risk agent + sot agent +
# netbox agent + deployment plan agent + ansible agent + full plan preview

if not AIDOS_READY:
    print("Skipping local multi-agent pass: preflight is BLOCKED. Fix missing files in AIDOS_INPUT_ROOT first.")
else:
    bundle = ingest_inputs(
        AIDOS_SURVEY_PATH,
        bom_path=AIDOS_BOM_PATH,
        workload_path=AIDOS_WORKLOAD_PATH,
        context_path=AIDOS_CONTEXT_PATH,
        network_layout_path=AIDOS_NETWORK_LAYOUT_PATH,
    )

    sot = build_canonical_sot(bundle)
    validation_report, missing_data_report = validate_sot(sot, observed_path=AIDOS_OBSERVED_PATH)
    netbox_payload = build_netbox_payload(sot)
    runbook = build_runbook_plan(sot, validation_report)
    ansible_playbook = build_ansible_playbook(runbook)
    ansible_bundle = build_ansible_bundle(
        runbook,
        architecture={
            "project": sot.project.model_dump(mode="json"),
            "intent": sot.intent.model_dump(mode="json"),
            "expected": sot.expected.model_dump(mode="json"),
            "site": sot.site.model_dump(mode="json"),
            "workload": sot.workload.model_dump(mode="json") if sot.workload else None,
            "netbox_payload": netbox_payload.model_dump(mode="json"),
        },
    )
    agentic_task_graph = build_agentic_task_graph(runbook)

    print("=" * 72)
    print("AIDOS MULTI-AGENT PASS")
    print("=" * 72)
    print(f"Deployment: {sot.intent.deployment_name}")
    print(f"Readiness: {validation_report.readiness}")
    print(f"Findings: critical={len(validation_report.critical)} warning={len(validation_report.warning)} passed={len(validation_report.passed)}")
    print(f"Missing-data items: {len(missing_data_report.items)}")
    print(
        f"NetBox payload: sites={len(netbox_payload.sites)} racks={len(netbox_payload.racks)} "
        f"devices={len(netbox_payload.devices)} vlans={len(netbox_payload.vlans)} "
        f"prefixes={len(netbox_payload.prefixes)} cables={len(netbox_payload.cables)}"
    )
    print(f"Runbook tasks: {len(runbook.tasks)}")
    print(f"Ansible bundle files: {len(ansible_bundle.keys())}")
    print(f"Task graph nodes: {len(agentic_task_graph.get('nodes', []))}")

    print("\nTask intents:")
    for task in runbook.tasks:
        print(f"- {task.id}: {task.intent} ({task.preferred_executor})")

In [ ]:
# Optional: write full AIDOS artifact set in one call.
# Set EXECUTE_WORKFLOW=True only when you want task execution attempts.

EXECUTE_WORKFLOW = False
SYNC_NETBOX = False
AUTO_APPROVE = False

if not AIDOS_READY:
    print("Skipping workflow writer: preflight is BLOCKED. Fix missing files in AIDOS_INPUT_ROOT first.")
else:
    artifacts = run_mvp_workflow(
        survey_path=AIDOS_SURVEY_PATH,
        output_dir=AIDOS_OUTPUT_DIR,
        bom_path=AIDOS_BOM_PATH,
        workload_path=AIDOS_WORKLOAD_PATH,
        context_path=AIDOS_CONTEXT_PATH,
        network_layout_path=AIDOS_NETWORK_LAYOUT_PATH,
        observed_path=AIDOS_OBSERVED_PATH,
        sync_netbox=SYNC_NETBOX,
        execute=EXECUTE_WORKFLOW,
        auto_approve=AUTO_APPROVE,
    )

    print("\nAIDOS artifact paths:")
    for name, value in artifacts.__dict__.items():
        print(f"- {name}: {value}")


In [ ]:
# Optional: RAG-grounded operator query over AIDOS artifacts after workflow run.
# Requires RAG services + indexed docs to be up if you want semantic external grounding.

AIDOS_OPERATOR_QUESTION = "What are my highest deployment blockers and what should I do next?"

try:
    rag_grounding = await collect_rag_grounding(AIDOS_OPERATOR_QUESTION, collection=globals().get("COLLECTION_NAME"))
    print("\nRAG grounding summary:")
    print(rag_grounding["summary"])
except Exception as exc:
    print(f"RAG grounding skipped: {exc}")
